# Data

In [1]:
import json

with open('data/result.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [2]:
import pandas as pd

df = pd.DataFrame(data['messages'])

In [3]:
from utils import preprocess_df

df = preprocess_df(df)

Загружено сообщений для анализа: 32738


# Pipeline

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(df['clean_text'].tolist(), show_progress_bar=True)
# np.save('embeddings.npy', embeddings)

# embeddings = np.load('embeddings.npy')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1024 [00:00<?, ?it/s]

In [5]:
import umap

reducer = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

embeddings_reduced = reducer.fit_transform(embeddings)

C:\Users\nikol\AppData\Local\Programs\Python\Python314\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [6]:
from sklearn.cluster import HDBSCAN

model = HDBSCAN()
df['cluster'] = model.fit_predict(embeddings_reduced)

C:\Users\nikol\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


# Analysis

In [7]:
df

,date,text,from_id,clean_text,cluster
3,2025-08-01 03:16:34,Какое у него тупое еблище,user1547888108,Какое у него тупое еблище,525
4,2025-08-01 03:16:36,Зацените,user1547888108,Зацените,272
7,2025-08-01 07:13:31,Доброе утро мальчики,user1547888108,Доброе утро мальчики,642
8,2025-08-01 07:23:09,Невозможно,user818491332,Невозможно,346
9,2025-08-01 07:23:49,Virgin whiskers,user818491332,Virgin whiskers,-1
...,...,...,...,...,...
42690,2026-01-31 17:30:21,пиздец,user492985890,пиздец,275
42691,2026-01-31 18:05:07,марк ты вообще мудак что ли.,user5386481314,марк ты вообще мудак что ли.,-1
42692,2026-01-31 18:14:58,ты даже в нерабочее время на страже я смотрю,user492985890,ты даже в нерабочее время на страже я смотрю,-1
42694,2026-01-31 20:37:21,шмальdestroyer,user877035234,шмальdestroyer,-1


In [22]:
df[df['cluster'] != -1][['clean_text', 'cluster']]

,clean_text,cluster
3,Какое у него тупое еблище,525
4,Зацените,272
7,Доброе утро мальчики,642
8,Невозможно,346
10,Ебать звучит круто,701
...,...,...
42683,Очень интересно,1073
42685,Где ты это находишь?,380
42686,это я записал,771
42690,пиздец,275


In [ ]:
df[df['cluster'] == 10]['clean_text']

In [ ]:
df[df['cluster'] == 12]['clean_text']

In [21]:
df[df['cluster'] == 43]['clean_text']

1064             а че ты так быстро
2328                        Быстрее
2404                        Быстрее
2424                  Быстро только
2579                        Быстрее
2599                        Быстрее
2767                  Быстро только
4086                  Быстро только
8605                        Быстрее
10666                      Дропнуть
13465              Титул мне быстро
19612                       биг брр
23202                 да он быстрый
31205    Что бы быстрее всë сделать
Name: clean_text, dtype: object

In [9]:
df['cluster'].value_counts().describe()

count     1349.000000
mean        24.268347
std        308.682429
min          5.000000
25%          7.000000
50%         10.000000
75%         17.000000
max      11318.000000
Name: count, dtype: float64